# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`

This notebook demonstrates how to explore, process, and visualize a dataset defined by a Croissant schema using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`. We'll print the dataset's name and description to confirm successful loading.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant metadata URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json"

# Load dataset
dataset = mlc.Dataset(croissant_url)

# View summary metadata
print(f"{dataset.metadata.name}: {dataset.metadata.description}")

## 2. Data Overview

Let's inspect the available record sets in the dataset. Each record set, field, and column are referenced by their Croissant `@id` for clarity and reproducibility.

In [ ]:
# List all record sets
print("Available Record Sets (by @id):")
for rs in dataset.metadata.record_sets:
    print(f"- Record Set ID: {rs.id}  |  Name: {rs.name}")

# For this dataset, let's focus on the main survey responses record set.
# We'll also enumerate fields for our main record set.

# Pick the main record set (usually demographics & scores)
record_set_id = dataset.metadata.record_sets[0].id  # e.g. 'kilifi_survey_responses.csv'

record_set = None
for rs in dataset.metadata.record_sets:
    if rs.id == record_set_id:
        record_set = rs
        break

assert record_set is not None, 'Could not find main record set.'

print(f"\nFields in Record Set '{record_set.name}' (by @id):")
for field in record_set.fields:
    # Field ID, Name, column id/type info
    print(f"  - Field ID: {field.id} | Name: {field.name} | Column: {field.column.id if hasattr(field, 'column') else 'N/A'} | Data type: {getattr(field, 'data_type', 'N/A')}")

## 3. Data Extraction

Load the main record set into a `pandas.DataFrame` for processing. All column names correspond to their Croissant column `@id` (as specified in the schema).

In [ ]:
# List of all record set IDs for potential further exploration
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]
dataframes = {}

for rid in record_set_ids:
    df = pd.DataFrame(dataset.records(record_set=rid))
    dataframes[rid] = df
    print(f"Loaded {len(df)} records for Record Set '{rid}'.")

main_df = dataframes[record_set_id]

print("\nColumns in the main DataFrame (by field/column @id):")
print(main_df.columns.tolist())
main_df.head()

## 4. Exploratory Data Analysis (EDA)

Apply data processing steps on the main DataFrame. We will:
- Select a quantitative (numeric) mental health score field (e.g., GAD-7 or PHQ-9 total score by its Croissant `@id`)
- Filter participants based on a threshold score
- Normalize the selected score
- Group by a demographic attribute such as `gender` (using its `@id`)

**Note:** Replace field IDs below with the actual Croissant field/column `@id` values as reported in the metadata.

In [ ]:
# Choose field IDs for columns (update based on schema listing if needed)
# For this hypothetical example, suppose the following IDs exist in the dataset:
# gad7_total_score: 'kilifi_survey_responses.csv/gad7_total_score'
# gender: 'kilifi_survey_responses.csv/gender'
# age: 'kilifi_survey_responses.csv/age'
# phq9_total_score: 'kilifi_survey_responses.csv/phq9_total_score'

numeric_field_id = None
group_field_id = None

# Try to infer likely score/numeric/gender IDs from available columns
for c in main_df.columns:
    if 'gad' in c and ('score' in c or 'total' in c):
        numeric_field_id = c
    elif 'phq' in c and ('score' in c or 'total' in c):
        # Use PHQ-9 if GAD-7 isn't found
        if numeric_field_id is None:
            numeric_field_id = c
    elif 'age' in c:
        group_field_id = c
    elif 'gender' in c:
        group_field_id = c

if numeric_field_id is None:
    print("Warning: Could not find a standard GAD-7 or PHQ-9 field by @id. Please check the printed schema overview above.")
else:
    print(f"Selected numeric field: {numeric_field_id}")
if group_field_id is None:
    print("Warning: Could not find typical group field (age/gender).")
else:
    print(f"Grouping field: {group_field_id}")

# Only continue if a numeric field is found
if numeric_field_id is not None and pd.api.types.is_numeric_dtype(main_df[numeric_field_id]):
    threshold = main_df[numeric_field_id].mean()
    filtered_df = main_df[main_df[numeric_field_id] > threshold].copy()

    print(f"\nFiltered records with {numeric_field_id} > {threshold:.2f}:")
    print(filtered_df.head())

    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()

    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, norm_col]].head())

    # Group by selected demographic field
    if group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"\nGrouped data by {group_field_id} (mean {numeric_field_id}):")
        print(grouped_df.head())
else:
    print("No numeric score field detected for EDA.")

## 5. Visualization

Let's visualize the distribution of the numeric mental health score and the mean score by demographic group (e.g., by gender or age groups).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of total scores
if numeric_field_id is not None and numeric_field_id in main_df.columns and pd.api.types.is_numeric_dtype(main_df[numeric_field_id]):
    plt.figure(figsize=(8,4))
    sns.histplot(main_df[numeric_field_id].dropna(), bins=12, kde=True, color='skyblue')
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    if group_field_id is not None and group_field_id in main_df.columns:
        plt.figure(figsize=(8,5))
        sns.boxplot(x=main_df[group_field_id], y=main_df[numeric_field_id])
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()
else:
    print("Visualization skipped: No numeric field found.")

## 6. Conclusion

* In this notebook, we loaded and explored the FAIR² Mental Health Survey Dataset using the `mlcroissant` library with full reference to Croissant schema entity `@id` fields.
* We demonstrated loading, field inspection, filtering, normalization, grouping, and visualization of data via `mlcroissant` and `pandas`.
* This approach is robust, reproducible, and links all processing steps to data provenance identifiers defined in the Croissant schema.

You can extend this notebook with additional analyses, joining on other record sets or exploring more advanced data science workflows, always referencing data elements by their `@id` for maximum clarity and FAIRness!